# LangGraph CLI

LangGraph 命令行界面包含在Docker中本地构建和运行 LangGraph 平台 API 服务器的命令。对于开发和测试，您可以使用 CLI 部署本地 API 服务器。

## 安装
- 确保已安装 Docker（例如docker --version）。
- 安装 CLI 包：
`pip install langgraph-cli` 或 `npm install -g @langchain/langgraph-cli`

- 运行命令`langgraph --help`或`npx @langchain/langgraph-cli --help`确认 CLI 正常工作。


## 配置文件

LangGraph CLI 需要一个遵循此架构的 JSON 配置文件。它包含以下属性：

> LangGraph CLI 默认使用当前目录中的配置文件`langgraph.json` 。

### Python版本
| 键 | 描述 | 
| :--- | :--- | 
| dependencies | 必需。LangGraph 平台 API 服务器的依赖项数组。依赖项可以是以下之一：<br>- 一个句点 ( ".")，它将查找本地 Python 包。<br> - pyproject.toml、setup.py或requirements.txt所在的目录路径。例如，如果requirements.txt位于项目目录的根目录，则指定"./"。如果它位于名为 的子目录中local_package，则指定"./local_package"。请勿指定字符串"requirements.txt"本身。<br> - Python 包名称。 | 
| graphs | 必需。从图表 ID 映射到已编译图表或创建图表的函数的定义路径。示例：<br> 1️⃣ `./your_package/your_file.py:variable`，其中variable是langgraph.graph.state.CompiledStateGraph <br>  2️⃣`./your_package/your_file.py:make_graph`，其中make_graph是一个函数，它接受一个配置字典（ ）并返回或 的langchain_core.runnables.RunnableConfig实例。有关更多详细信息，请参阅如何在运行时重建图表。`langgraph.graph.state.StateGraphlanggraph` `.graph.state.CompiledStateGraph` | 
| auth | （v0.0.11 新增）身份验证配置包含身份验证处理程序的路径。例如：`./your_package/auth.py:auth`，其中`auth`是 的一个实例`langgraph_sdk.Auth`。详情请参阅身份验证指南。 | 
| base_image | 可选。用于 LangGraph API 服务器的基础镜像。默认为`langchain/langgraph-api`或`langchain/langgraphjs-api`。使用此选项可以将你的构建版本固定到特定版本的 `langgraph API`，例如`"langchain/langgraph-server:0.2"`。更多详情请参阅`https://hub.docker.com/r/langchain/langgraph-server/tagslanggraph-cli==0.2.8` 。（新增于） | 
| image_distro | 可选。基础映像的 Linux 发行版。必须是`"debian"`或`"wolfi"`。如果省略，则默认为`"debian"`。可在 中使用`langgraph-cli>=0.2.11`。 | 
| env | `.env` 文件的路径或环境变量与其值的映射。 | 
| store | 用于向 BaseStore 添加语义搜索和/或生存时间 (TTL) 的配置。包含以下字段：<br>-index（可选）：使用字段 、 和可选 进行语义搜索索引embed的dims配置fields。<br>-ttl（可选）：项目过期配置。一个包含以下可选字段的对象：（refresh_on_read布尔值，默认为true）、default_ttl（浮点数，以分钟为单位的寿命，默认为无过期时间）和sweep_interval_minutes（整数，检查过期项目的频率，默认为无清除）。 | 
| ui | 可选。代理发出的 UI 组件的命名定义，每个定义指向一个 JS/TS 文件。（添加于langgraph-cli==0.1.84） | 
| python_version | 3.11、3.12、 或3.13。默认为3.11。 | 
| node_version | 指定node_version: 20使用LangGraph.js。 | 
| pip_config_file | 配置文件的路径pip。 | 
| pip_installer | (在 v0.3 中添加）可选。Python 软件包安装程序选择器。可以设置为 "auto"、"pip "或 "uv"。从 0.3 版开始，默认策略是运行 uv pip，它通常能提供更快的编译速度，同时仍可直接替换。在 uv 无法处理依赖关系图或 pyproject.toml 结构的罕见情况下，请在此处指定 "pip"，以恢复到先前的行为。 | 
| keep_pkg_tools | （v0.3.4 新增）可选。控制是否在最终镜像中保留 Python 打包工具（pip、setuptools、wheel ）。可接受的值：<br>-true：保留所有三个工具（跳过卸载）。<br>-false/省略：卸载所有三个工具（默认行为）。<br>-list[str]：要保留的工具名称。每个值必须是“pip”、  “setuptools”或“wheel”之一。。默认情况下，这三个工具均被卸载。 | 
| dockerfile_lines | 从父镜像导入后要添加到 Dockerfile 的附加行数组。 | 
| checkpointer | 检查点的配置。包含一个ttl对象字段，该对象具有以下键：<br> 1️⃣strategy：如何处理过期的检查点（例如"delete"）。<br>2️⃣sweep_interval_minutes：检查过期检查点的频率（整数）。<br>3️⃣default_ttl：检查点的默认生存时间，以分钟为单位（整数）。定义在应用指定策略之前检查点的保留时间。 | 
| http | HTTP 服务器配置包含以下字段：<br>1️⃣app：自定义 Starlette/FastAPI 应用的路径（例如"./src/agent/webapp.py:app"）。请参阅自定义路线指南。<br>2️⃣cors：CORS 配置，包含allow_origins、allow_methods、allow_headers等字段。<br>3️⃣configurable_headers：定义哪些请求标头需要排除或包含为运行的可配置值。<br>4️⃣disable_assistants：禁用/assistants路线<br>5️⃣disable_mcp：禁用/mcp路线<br>6️⃣disable_meta：禁用/ok、/info、/metrics和/docs路线<br>7️⃣disable_runs：禁用/runs路线<br>8️⃣disable_store：禁用/store路线<br>9️⃣disable_threads：禁用/threads路线<br>🔟disable_ui：禁用/ui路线<br>disable_webhooks：在所有路由中禁用运行完成时的 webhook 调用<br>mount_prefix：已安装路由的前缀（例如“/my-deployment/api”） | 




### Graphs的无需重建示例
在标准 `LangGraph API` 配置中，服务器使用在 `openai_agent.py` 顶层定义的已编译图形实例，如下所示：

要让服务器知道您的图，您需要在 LangGraph API 配置（langgraph.json）中指定包含` CompiledStateGraph` 实例的变量的路径，例如:

```json
{
    "dependencies": ["."],
    "graphs": {
        "openai_agent": "./openai_agent.py:agent",
    },
    "env": "./.env"
}
```

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessageGraph

model = ChatOpenAI(temperature=0)

graph_workflow = MessageGraph()

graph_workflow.add_node("agent", model)
graph_workflow.add_edge("agent", END)
graph_workflow.add_edge(START, "agent")

agent = graph_workflow.compile()

### Graphs 重建
为了使图表在每次使用自定义配置重新运行时重建，您需要重写openai_agent.py代码，提供一个接受配置并返回图表（或编译后的图表）实例的函数。假设我们想为用户 ID 为“1”的用户返回现有图表，并为其他用户返回工具调用代理。我们可以进行openai_agent.py如下修改：


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, MessageGraph
from langgraph.graph.state import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableConfig


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


model = ChatOpenAI(temperature=0)

def make_default_graph():
    """Make a simple LLM agent"""
    graph_workflow = StateGraph(State)
    def call_model(state):
        return {"messages": [model.invoke(state["messages"])]}

    graph_workflow.add_node("agent", call_model)
    graph_workflow.add_edge("agent", END)
    graph_workflow.add_edge(START, "agent")

    agent = graph_workflow.compile()
    return agent


def make_alternative_graph():
    """Make a tool-calling agent"""

    @tool
    def add(a: float, b: float):
        """Adds two numbers."""
        return a + b

    tool_node = ToolNode([add])
    model_with_tools = model.bind_tools([add])
    def call_model(state):
        return {"messages": [model_with_tools.invoke(state["messages"])]}

    def should_continue(state: State):
        if state["messages"][-1].tool_calls:
            return "tools"
        else:
            return END

    graph_workflow = StateGraph(State)

    graph_workflow.add_node("agent", call_model)
    graph_workflow.add_node("tools", tool_node)
    graph_workflow.add_edge("tools", "agent")
    graph_workflow.add_edge(START, "agent")
    graph_workflow.add_conditional_edges("agent", should_continue)

    agent = graph_workflow.compile()
    return agent


# this is the graph making function that will decide which graph to
# build based on the provided config
def make_graph(config: RunnableConfig):
    user_id = config.get("configurable", {}).get("user_id")
    # route to different graph state / structure based on the user ID
    if user_id == "1":
        return make_default_graph()
    else:
        return make_alternative_graph()

最后，您需要在langgraph.json中指定制图功能的路径（`make_graph`）：

```json
{
    "dependencies": ["."],
    "graphs": {
        "openai_agent": "./openai_agent.py:make_graph",
    },
    "env": "./.env"
}
```

## 基本配置¶
```json
{
  "dependencies": ["."],
  "graphs": {
    "chat": "./chat/graph.py:graph"
  }
}
```

### 使用 Wolfi 基础镜像¶
您可以使用字段指定基础镜像的 Linux 发行版image_distro。有效选项为debian或wolfi。推荐使用 Wolfi，因为它提供更小、更安全的镜像。该选项在 中可用langgraph-cli>=0.2.11。

```json

{
  "dependencies": ["."],
  "graphs": {
    "chat": "./chat/graph.py:graph"
  },
  "image_distro": "wolfi"
}
```


### 向商店添加语义搜索¶
所有部署都附带一个基于数据库的 BaseStore。添加“索引”配置langgraph.json将启用部署中 BaseStore 的语义搜索功能。

配置index.fields决定了要嵌入文档的哪些部分：
- 如果省略或设置为["$"]，则整个文档将被嵌入
- 要嵌入特定字段，请使用 JSON 路径表示法：["metadata.title", "content.text"]
- 缺少指定字段的文档仍将被存储，但不会有这些字段的嵌入
- put您仍然可以使用index参数覆盖在特定项目上嵌入哪些字段


```json

{
  "dependencies": ["."],
  "graphs": {
    "memory_agent": "./agent/graph.py:graph"
  },
  "store": {
    "index": {
      "embed": "openai:text-embedding-3-small",
      "dims": 1536,
      "fields": ["$"]
    }
  }
}
```

- openai:text-embedding-3-large: 3072
- openai:text-embedding-3-small: 1536
- openai:text-embedding-ada-002: 1536
- cohere:embed-english-v3.0: 1024
- cohere:embed-english-light-v3.0: 384
- cohere:embed-multilingual-v3.0: 1024
- cohere:embed-multilingual-light-v3.0: 384



### 具有自定义嵌入函数的语义搜索¶
如果您想使用自定义嵌入函数进行语义搜索，则可以将路径传递给自定义嵌入函数：

```json
{
  "dependencies": ["."],
  "graphs": {
    "memory_agent": "./agent/graph.py:graph"
  },
  "store": {
    "index": {
      "embed": "./embeddings.py:embed_texts",
      "dims": 768,
      "fields": ["text", "summary"]
    }
  }
}
```

商店配置中的字段`embed`可以引用一个自定义函数，该函数接受一个字符串列表并返回一个嵌入列表。示例实现：

In [ ]:
# embeddings.py
def embed_texts(texts: list[str]) -> list[list[float]]:
    """Custom embedding function for semantic search."""
    # Implementation using your preferred embedding model
    return [[0.1, 0.2, ...] for _ in texts]  # dims-dimensional vectors

### 添加自定义身份验证

```json
{
  "dependencies": ["."],
  "graphs": {
    "chat": "./chat/graph.py:graph"
  },
  "auth": {
    "path": "./auth.py:auth",
    "openapi": {
      "securitySchemes": {
        "apiKeyAuth": {
          "type": "apiKey",
          "in": "header",
          "name": "X-API-Key"
        }
      },
      "security": [{ "apiKeyAuth": [] }]
    },
    "disable_studio_auth": false
  }
}
```

有关详细信息，请参阅身份验证概念指南，有关该过程的实际操作，请参阅设置自定义身份验证指南。

### 配置存储项的有效时间 (TTL) 

您可以使用 `store.ttl` 密钥为 `BaseStore` 中的项目/内存配置默认数据过期时间。这决定了项目在最后一次被访问后的保留时间（读取时可能会根据 `refresh_on_read` 刷新计时器）。请注意，可以通过修改 `get`、`search` 等调用中的相应参数来覆盖这些默认值。

- `refresh_on_read`：如果设置为 `true`（默认值），则通过 `get` 或 `search` 访问项目时会重置其过期计时器。设为 false 则只在写入（put）时刷新 TTL。
- `default_ttl`：项目的默认寿命，以分钟为单位。如果未设置，项目默认不会过期。
- `sweep_interval_minutes`：系统运行后台进程删除过期项目的频率（以分钟为单位）。如果未设置，则不会自动进行清扫。

下面的示例启用了 7 天的 TTL（10080 分钟），读取时刷新，每小时扫描一次：

```json

{
  "dependencies": ["."],
  "graphs": {
    "memory_agent": "./agent/graph.py:graph"
  },
  "store": {
    "ttl": {
      "refresh_on_read": true,
      "sweep_interval_minutes": 60,
      "default_ttl": 10080 
    }
  }
}
```

### 配置检查点生存时间 (TTL)

可以使用 `checkpointer` 键配置检查点的存活时间（TTL）。这决定了检查点数据在根据指定策略（如删除）自动处理之前的保留时间。ttl 配置是一个包含以下内容的对象：
- `strategy`：对过期检查点执行的作（当前“delete”是唯一接受的选项）。
- `sweep_interval_minutes`（扫描间隔分钟数）：系统检查过期检查点的频率（分钟）。
- `default_ttl`：检查点的默认寿命，以分钟为单位。

下面是一个设置默认 TTL 为 30 天（43200 分钟）的示例：
```json
{
  "dependencies": ["."],
  "graphs": {
    "chat": "./chat/graph.py:graph"
  },
  "checkpointer": {
    "ttl": {
      "strategy": "delete",
      "sweep_interval_minutes": 10,
      "default_ttl": 43200
    }
  }
}
```

在此示例中，将删除超过30天的检查站，并且支票每10分钟进行一次。

## 命令
LangGraph CLI 的基本命令是`langgraph`。

```shell
langgraph [OPTIONS] COMMAND [ARGS]
```

### dev
以开发模式运行 LangGraph API 服务器，并支持热重载和调试功能。此轻量级服务器无需安装 Docker，适用于开发和测试。状态将持久保存到本地目录。


#### 安装
此命令需要安装“inmem”附加组件：

`pip install -U "langgraph-cli[inmem]"`

#### 用法
`langgraph dev [OPTIONS]`


| 选项 | 默认 | 描述 |
| :--- | :--- | :--- |
| －c，－－config | langgraph．json FILE| 声明依赖项、图表和环境变量的配置文件的路径 |
| －－host TEXT | 127．0．0．1 | 绑定服务器的主机 |
| －－port INTEGER| 2024 | 绑定服务器的端口 |
| －－no－reload |  | 禁用自动重新加载 |
| －－n－jobs－per－worker INTEGER |  | 每个 worker 的作业数量。默认值为 10 |
| －－debug－port INTEGER|  | 调试器监听的端口 |
| －－wait－for－client | False | 在启动服务器之前等待调试器客户端连接到调试端口 |
| －－no－browser |  | 服务器启动时跳过自动打开浏览器 |
| --studio-url TEXT |  | 需要连接的 LangGraph Studio 实例的 URL。默认为https://smith.langchain.com |
| --allow-blocking | False | 不要在代码中引发同步 I/O 阻塞操作的错误（添加于0.2.6） |
| --tunnel | False | 通过公共隧道（Cloudflare）公开本地服务器，以便远程前端访问。这可以避免Safari等浏览器或网络阻塞本地主机连接的问题。 |
| --help |    | 显示命令文档 |


### build
构建 LangGraph 平台 API 服务器 Docker 镜像。

#### 用法
`langgraph build [OPTIONS]`

| 选项 | 默认 | 描述 |
| :--- | :--- | :--- |
| --platform TEXT | 构建 Docker 镜像的目标平台。例如：langgraph build --platform linux/amd64,linux/arm64 |
| -t, --tag TEXT | 必填。Docker 镜像的标签。例如：langgraph build -t my-image |
| --pull / --no-pull | --pull | 使用最新的远程 Docker 镜像进行构建。用于--no-pull使用本地构建的镜像运行 LangGraph 平台 API 服务器。 |
| -c, --config FILE |langgraph.json   | 声明依赖项、图表和环境变量的配置文件的路径。 |
| --help |    | 显示命令文档。 |


### up
启动 LangGraph API 服务器。本地测试需要 LangSmith API 密钥，并允许访问 LangGraph 平台。生产使用需要许可证密钥。

#### 用法
`langgraph up [OPTIONS]`

| 选项 | 默认 | 描述 |
| :--- | :--- | :--- |
| --wait |  | 等待服务启动后再返回。暗示 —detach  |
| --base-image TEXT | langchain/langgraph-api | 用于 LangGraph API 服务器的基础镜像。使用版本标签固定到特定版本。 |
| --image TEXT |  | 用于 langgraph-api 服务的 Docker 镜像。如果指定，则跳过构建并直接使用此镜像。 |
| --postgres-uri TEXT | 本地数据库 | 用于数据库的 Postgres URI。 |
| --watch |  | 文件更改时重新启动 |
| --debugger-base-url TEXT | http://127.0.0.1:[PORT] | 调试器用于访问 LangGraph API 的 URL。 |
| --debugger-port INTEGER |  | 将调试器镜像拉到本地并在指定端口上提供 UI |
| --verbose |  | 显示服务器日志的更多输出。 |
| -c, --config FILE | langgraph.json | 声明依赖项、图表和环境变量的配置文件的路径。 |
| -d, --docker-compose FILE |  | 包含要启动的附加服务的 docker-compose.yml 文件的路径。 |
| -p, --port INTEGER | 8123 | 要公开的端口。例如：langgraph up --port 8000 |
|  --pull / --no-pull | pull | 拉取最新镜像。用于--no-pull使用本地构建的镜像运行服务器。例如：langgraph up --no-pull |
| --recreate / --no-recreate | no-recreate | 即使容器的配置和镜像没有改变，也可以重新创建容器 |
| --help |  | 显示命令文档。 |


### dockerfile
生成用于构建 LangGraph 平台 API 服务器 Docker 镜像的 Dockerfile。
#### 用法
`langgraph dockerfile [OPTIONS] SAVE_PATH`

| 选项 | 默认 | 描述 |
| :--- | :--- | :--- |
|-c, --config FILE | langgraph.json | 声明依赖项、图表和环境变量的配置文件的路径。  |
| --help |  | 显示此消息并退出。 |


#### 例子：
`langgraph dockerfile -c langgraph.json Dockerfile`


这将生成一个类似于以下内容的 Dockerfile：

```json
FROM langchain/langgraph-api:3.11

ADD ./pipconf.txt /pipconfig.txt

RUN PIP_CONFIG_FILE=/pipconfig.txt PYTHONDONTWRITEBYTECODE=1 pip install --no-cache-dir -c /api/constraints.txt langchain_community langchain_anthropic langchain_openai wikipedia scikit-learn

ADD ./graphs /deps/__outer_graphs/src
RUN set -ex && \
    for line in '[project]' \
                'name = "graphs"' \
                'version = "0.1"' \
                '[tool.setuptools.package-data]' \
                '"*" = ["**/*"]'; do \
        echo "$line" >> /deps/__outer_graphs/pyproject.toml; \
    done

RUN PIP_CONFIG_FILE=/pipconfig.txt PYTHONDONTWRITEBYTECODE=1 pip install --no-cache-dir -c /api/constraints.txt -e /deps/*

ENV LANGSERVE_GRAPHS='{"agent": "/deps/__outer_graphs/src/agent.py:graph", "storm": "/deps/__outer_graphs/src/storm.py:graph"}'
```